# 레슨 10 — 통합 프로젝트 정답지

> 교사·관리자 전용. 학생에게 배포하지 않는다.

이 정답지는 학생용 `mission.md` 의 문제 1~15와 번호가 1:1로 대응한다. 이번 강의의 핵심은 단일 함수 연습이 아니라, 데이터 품질 확인부터 발표 결론까지 하나의 분석 흐름을 완성하는 것이다.

## 환경 셀

In [ ]:
import os
import pandas as pd
import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

IS_COLAB = "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ
if IS_COLAB:
    DATA_BASE = "https://raw.githubusercontent.com/Kevin-innovation/jupyter-lecture/main/python-data-analysis/lectures/10/data"
else:
    DATA_BASE = "./data"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

print("pandas:", pd.__version__)
print("data base:", DATA_BASE)

---

## 문제 1 정답 — 프로젝트 데이터 불러오기

In [ ]:
df = pd.read_csv(f"{DATA_BASE}/city_learning_programs.csv")

print("shape:", df.shape)
print("columns:", list(df.columns))
print("dtypes:")
print(df.dtypes)
print("앞 5행:")
print(df.head())
print("뒤 3행:")
print(df.tail(3))
print("결측치:")
print(df.isna().sum())
print("중복 program_id:", df["program_id"].duplicated().sum())

### 왜 이 코드가 정답인지

통합 프로젝트에서는 어떤 분석을 할지 정하기 전에 데이터 품질을 확인해야 한다. `shape`, `columns`, `dtypes`, 결측치, 중복 ID를 확인하면 프로그램 단위 데이터가 한 행에 하나씩 들어 있는지 판단할 수 있다. `program_id` 중복이 없으면 프로그램 수 집계가 행 수와 같은 기준으로 진행된다.

**예상 핵심값**

| 항목 | 값 |
|---|---:|
| 행 수 | 720 |
| 열 수 | 9 |
| 중복 ID | 0개 |

---

## 문제 2 정답 — 날짜와 기간 열 만들기

In [ ]:
df["start_date"] = pd.to_datetime(df["start_date"])
df["start_month"] = df["start_date"].dt.strftime("%Y-%m")
df["start_quarter"] = df["start_date"].dt.to_period("Q").astype(str)

print("시작일 범위:", df["start_date"].min().date(), "~", df["start_date"].max().date())
monthly_program_count = df["start_month"].value_counts().sort_index()
print(monthly_program_count)
print(df[["program_id", "start_date", "start_month", "start_quarter"]].head())

### 왜 이 코드가 정답인지

프로그램 시작일은 월별 운영량과 분기별 운영 흐름을 보는 기준이 된다. 날짜형으로 변환한 뒤 `start_month`, `start_quarter` 를 만들면 월·분기 단위 그룹화가 쉬워진다. 월별 프로그램 수를 출력하면 특정 월에 운영이 몰렸는지 먼저 확인할 수 있다.

**예상 확인 포인트**

```text
start_month 는 YYYY-MM 문자열
start_quarter 는 2025Q1 같은 분기 문자열
```

---

## 문제 3 정답 — 프로젝트 품질 점검

In [ ]:
numeric_cols = ["participants", "budget", "completion_rate", "satisfaction"]
print(df[numeric_cols].describe())

invalid_participants = df[df["participants"] <= 0]
invalid_budget = df[df["budget"] <= 0]
invalid_completion = df[~df["completion_rate"].between(0, 1)]
invalid_satisfaction = df[~df["satisfaction"].between(1, 5)]

print("참여자 수 오류:", len(invalid_participants))
print("예산 오류:", len(invalid_budget))
print("완료율 범위 오류:", len(invalid_completion))
print("만족도 범위 오류:", len(invalid_satisfaction))

### 왜 이 코드가 정답인지

프로젝트형 데이터는 값의 범위가 정상인지 확인해야 이후 KPI 계산이 의미를 가진다. 참여자 수와 예산은 0보다 커야 하고, 완료율은 0~1, 만족도는 1~5 범위여야 한다. 범위 오류가 없으면 파생 지표를 만들 때 별도 보정 없이 진행할 수 있다.

**채점 기준**

| 열 | 정상 기준 |
|---|---|
| participants | 0보다 큼 |
| budget | 0보다 큼 |
| completion_rate | 0 이상 1 이하 |
| satisfaction | 1 이상 5 이하 |

---

## 문제 4 정답 — 핵심 KPI 만들기

In [ ]:
df["cost_per_participant"] = df["budget"] / df["participants"]
df["effective_completions"] = df["participants"] * df["completion_rate"]
df["completion_band"] = pd.cut(
    df["completion_rate"],
    bins=[0, 0.7, 0.85, 1.0],
    labels=["low", "mid", "high"],
    include_lowest=True,
)

print(df[["program_id", "participants", "budget", "completion_rate", "cost_per_participant", "effective_completions", "completion_band"]].head())
print(df[["cost_per_participant", "effective_completions", "completion_rate", "satisfaction"]].agg(["mean", "median"]).round(2))

### 왜 이 코드가 정답인지

참여자 수, 예산, 완료율만 따로 보면 프로그램의 효율을 판단하기 어렵다. `cost_per_participant` 는 비용 효율을, `effective_completions` 는 실제 완료자 규모를 추정한다. `completion_band` 는 완료율을 구간으로 나누어 뒤에서 우수·점검 기준을 설명하기 쉽게 만든다.

**예상 확인 포인트**

```text
completion_band 값은 low, mid, high 중 하나
```

---

## 문제 5 정답 — 프로그램 유형별 요약

In [ ]:
program_summary = (
    df.groupby("program_type")
    .agg(
        programs=("program_id", "count"),
        participants=("participants", "sum"),
        effective_completions=("effective_completions", "sum"),
        budget=("budget", "sum"),
        completion_rate=("completion_rate", "mean"),
        satisfaction=("satisfaction", "mean"),
        cost_per_participant=("cost_per_participant", "median"),
    )
    .sort_values("participants", ascending=False)
)

print(program_summary.round(2))
print("참여자 수 1위 프로그램 유형:", program_summary["participants"].idxmax())

### 왜 이 코드가 정답인지

프로그램 유형별 요약은 어떤 교육 분야가 가장 많은 학생에게 도달했는지 보여준다. 합계가 필요한 지표와 평균 또는 중앙값이 필요한 지표를 분리해 집계해야 해석이 정확하다. 참여자 수 기준으로 정렬하면 규모가 큰 유형부터 확인할 수 있다.

**수업 중 확인 질문**

```text
참여자 수 1위 유형이 만족도도 1위인가?
```

---

## 문제 6 정답 — 지역별 요약

In [ ]:
district_summary = (
    df.groupby("district")
    .agg(
        programs=("program_id", "count"),
        participants=("participants", "sum"),
        budget=("budget", "sum"),
        completion_rate=("completion_rate", "mean"),
        satisfaction=("satisfaction", "mean"),
        cost_per_participant=("cost_per_participant", "mean"),
    )
    .sort_values("participants", ascending=False)
)

print(district_summary.round(2))
print("참여자 수 1위 지역:", district_summary["participants"].idxmax())
print("만족도 1위 지역:", district_summary["satisfaction"].idxmax())

### 왜 이 코드가 정답인지

지역별 요약은 운영 자원이 어느 지역에 많이 투입되고 어떤 지역의 만족도가 높은지 비교하게 해준다. 참여자 수와 만족도는 다른 질문에 답하므로 1위가 다를 수 있다. 이 차이를 확인해야 "규모가 큰 지역"과 "품질이 높은 지역"을 분리해 말할 수 있다.

**채점 기준**

| 기준 | 해석 |
|---|---|
| participants | 도달 규모 |
| satisfaction | 체감 품질 |
| cost_per_participant | 비용 효율 |

---

## 문제 7 정답 — 대상 그룹별 요약

In [ ]:
target_summary = (
    df.groupby("target_group")
    .agg(
        programs=("program_id", "count"),
        participants=("participants", "sum"),
        effective_completions=("effective_completions", "sum"),
        budget=("budget", "sum"),
        completion_rate=("completion_rate", "mean"),
        satisfaction=("satisfaction", "mean"),
        cost_per_participant=("cost_per_participant", "median"),
    )
    .sort_values("completion_rate", ascending=False)
)

print(target_summary.round(2))
print("완료율 평균 1위 대상 그룹:", target_summary["completion_rate"].idxmax())

### 왜 이 코드가 정답인지

대상 그룹별 요약은 초등, 중등, 고등 등 학습 대상에 따라 성과가 다르게 나타나는지 확인하는 단계다. 완료율 평균이 높은 그룹은 수업 난이도나 운영 방식이 잘 맞았을 가능성이 있다. 동시에 참여자 수와 예산도 함께 봐야 표본이 작은 그룹을 과대평가하지 않는다.

**주의할 점**

```text
완료율 1위 그룹이 항상 운영 우선순위 1위는 아니다.
```

---

## 문제 8 정답 — 지역 × 프로그램 유형 피벗

In [ ]:
district_program_pivot = pd.pivot_table(
    df,
    values="participants",
    index="district",
    columns="program_type",
    aggfunc="sum",
    fill_value=0,
)

print(district_program_pivot)
print("피벗 전체 참여자 수:", int(district_program_pivot.to_numpy().sum()))
print("원본 전체 참여자 수:", int(df["participants"].sum()))

stacked_participants = district_program_pivot.stack()
print("참여자 최대 조합:", stacked_participants.idxmax(), int(stacked_participants.max()))

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(district_program_pivot, annot=True, fmt=".0f", cmap="Blues", ax=ax)
ax.set_title("Participants: district x program type")
fig.tight_layout()
plt.close(fig)

### 왜 이 코드가 정답인지

피벗 테이블은 지역과 프로그램 유형이라는 두 범주를 동시에 비교하는 데 적합하다. 피벗 전체 합계와 원본 참여자 수 합계를 비교하면 집계가 누락 없이 이루어졌는지 검증할 수 있다. 히트맵은 어느 지역-유형 조합에 참여자가 집중되었는지 빠르게 보여준다.

**검증 기준**

```text
피벗 전체 참여자 수 == 원본 전체 참여자 수
```

---

## 문제 9 정답 — 완료율 × 만족도 관점의 우수 프로그램

In [ ]:
strong_programs = df[(df["completion_rate"] >= 0.85) & (df["satisfaction"] >= 4.5)]
review_programs = df[(df["completion_rate"] < 0.7) | (df["satisfaction"] < 3.5)]

print("우수 프로그램 수:", len(strong_programs))
print("점검 대상 프로그램 수:", len(review_programs))

print("우수 프로그램 상위:")
print(strong_programs.sort_values(["satisfaction", "completion_rate"], ascending=False).head(10))

print("점검 대상 프로그램 상위:")
print(review_programs.sort_values(["completion_rate", "satisfaction"]).head(10))

### 왜 이 코드가 정답인지

완료율과 만족도를 함께 사용하면 단순 참여자 수가 아니라 학습 품질 관점의 프로그램을 찾을 수 있다. 우수 프로그램은 두 기준을 모두 만족해야 하므로 `&` 를 사용하고, 점검 대상은 둘 중 하나라도 낮으면 확인해야 하므로 `|` 를 사용한다. 이 기준은 뒤 결론에서도 같은 표현으로 유지해야 한다.

**기준 요약**

| 그룹 | 조건 |
|---|---|
| strong_programs | 완료율 0.85 이상 그리고 만족도 4.5 이상 |
| review_programs | 완료율 0.7 미만 또는 만족도 3.5 미만 |

---

## 문제 10 정답 — 효율 점수 만들기

In [ ]:
df["completion_score"] = df["completion_rate"] * 100
df["satisfaction_score"] = df["satisfaction"] * 20

cost_min = df["cost_per_participant"].min()
cost_max = df["cost_per_participant"].max()
df["cost_score"] = 100 - ((df["cost_per_participant"] - cost_min) / (cost_max - cost_min) * 100)

df["efficiency_score"] = (
    df["completion_score"] * 0.4
    + df["satisfaction_score"] * 0.4
    + df["cost_score"] * 0.2
)

top_efficiency = df.sort_values("efficiency_score", ascending=False).head(10)
print(top_efficiency[["program_id", "district", "program_type", "target_group", "completion_rate", "satisfaction", "cost_per_participant", "efficiency_score"]].round(2))

### 왜 이 코드가 정답인지

효율 점수는 완료율, 만족도, 비용 효율을 같은 0~100 방향으로 맞춘 뒤 가중합으로 만든다. 비용은 낮을수록 좋은 지표이므로 `100 - 정규화값` 으로 방향을 뒤집어야 한다. 가중치는 운영 판단을 위한 예시이므로 절대 점수가 아니라 비교 기준으로 설명해야 한다.

**주의할 점**

```text
효율 점수는 의사결정 보조 지표이지 유일한 정답이 아니다.
```

---

## 문제 11 정답 — 예산과 성과 관계 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.scatterplot(data=df, x="budget", y="participants", hue="program_type", alpha=0.7, ax=axes[0])
axes[0].set_title("Budget vs participants")

sns.scatterplot(data=df, x="cost_per_participant", y="satisfaction", hue="program_type", alpha=0.7, ax=axes[1])
axes[1].set_title("Cost per participant vs satisfaction")
fig.tight_layout()
plt.close(fig)

corr_cols = ["participants", "budget", "completion_rate", "satisfaction", "cost_per_participant", "efficiency_score"]
project_corr = df[corr_cols].corr()
print(project_corr.round(3))

satisfaction_corr = project_corr["satisfaction"].drop("satisfaction").abs().sort_values(ascending=False)
print("만족도와 가장 함께 움직인 지표:", satisfaction_corr.index[0], f"{satisfaction_corr.iloc[0]:.3f}")

### 왜 이 코드가 정답인지

산점도는 예산과 참여자 수, 비용과 만족도처럼 두 숫자 지표의 관계를 볼 때 적합하다. 색을 프로그램 유형으로 나누면 유형별로 다른 패턴이 있는지 확인할 수 있다. 상관계수 표는 차트에서 본 관계를 숫자로 검증하는 보조 자료다.

**해석 주의**

```text
예산과 성과의 상관이 낮아도 예산이 중요하지 않다는 뜻은 아니다.
```

---

## 문제 12 정답 — 월별 운영 흐름

In [ ]:
monthly_ops = (
    df.groupby("start_month")
    .agg(
        programs=("program_id", "count"),
        participants=("participants", "sum"),
        budget=("budget", "sum"),
        completion_rate=("completion_rate", "mean"),
        satisfaction=("satisfaction", "mean"),
    )
    .sort_index()
)

print(monthly_ops.round(2))

fig, ax = plt.subplots(figsize=(8, 3))
sns.lineplot(data=monthly_ops.reset_index(), x="start_month", y="participants", marker="o", ax=ax)
ax.set_title("Monthly participants")
ax.set_xlabel("Month")
ax.set_ylabel("Participants")
fig.tight_layout()
plt.close(fig)

print("프로그램 수 1위 월:", monthly_ops["programs"].idxmax(), int(monthly_ops["programs"].max()))

### 왜 이 코드가 정답인지

월별 운영 흐름은 프로그램이 특정 시기에 몰리는지, 참여자 수가 어느 달에 늘어나는지 보여준다. `start_month` 로 그룹화하면 날짜 인덱스를 따로 설정하지 않아도 월별 요약을 만들 수 있다. 프로그램 수 1위 월을 출력하면 운영 집중 시점을 발표에 바로 사용할 수 있다.

**수업 중 질문**

```text
프로그램 수가 많은 월과 참여자 수가 많은 월이 같은가?
```

---

## 문제 13 정답 — 프로젝트 대시보드 표

In [ ]:
project_dashboard = pd.DataFrame(
    [
        {"metric": "programs", "value": len(df)},
        {"metric": "participants", "value": df["participants"].sum()},
        {"metric": "budget", "value": df["budget"].sum()},
        {"metric": "avg_completion_rate", "value": df["completion_rate"].mean()},
        {"metric": "avg_satisfaction", "value": df["satisfaction"].mean()},
        {"metric": "avg_cost_per_participant", "value": df["cost_per_participant"].mean()},
        {"metric": "strong_programs", "value": len(strong_programs)},
        {"metric": "review_programs", "value": len(review_programs)},
    ]
)

print(project_dashboard)

### 왜 이 코드가 정답인지

대시보드 표는 발표 첫 화면에서 전체 규모와 핵심 상태를 보여주는 요약이다. 전체 프로그램 수, 참여자 수, 예산, 평균 완료율, 만족도, 비용, 우수·점검 대상 수가 있으면 프로젝트의 큰 그림을 설명할 수 있다. 여러 값을 하나의 DataFrame으로 만들면 리포트나 슬라이드로 옮기기 쉽다.

**채점 기준**

```text
전체 규모 + 품질 평균 + 비용 효율 + 우수/점검 대상 수 포함
```

---

## 문제 14 정답 — 발표용 인사이트 5개 작성

In [ ]:
top_program_type = program_summary["participants"].idxmax()
top_district = district_summary["participants"].idxmax()
top_target = target_summary["completion_rate"].idxmax()
top_efficiency_program = top_efficiency.iloc[0]

insights = [
    f"프로그램 유형 기준 참여자 수는 {top_program_type} 유형이 가장 많습니다.",
    f"지역 기준 참여자 수는 {top_district} 지역이 가장 많습니다.",
    f"대상 그룹 기준 완료율은 {top_target} 그룹이 가장 높습니다.",
    f"효율 점수 1위 프로그램은 {top_efficiency_program['program_id']}입니다.",
    f"점검 대상 프로그램은 {len(review_programs)}개이므로 완료율 또는 만족도 기준으로 우선 확인합니다.",
]

for i, insight in enumerate(insights, start=1):
    print(f"{i}. {insight}")

### 왜 이 코드가 정답인지

발표용 인사이트는 표의 1위 값을 그대로 문장으로 바꾸는 단계다. 프로그램 유형, 지역, 대상 그룹, 효율, 점검 대상이라는 서로 다른 관점을 하나씩 포함하면 발표가 한 기준에 치우치지 않는다. 마지막 문장에 다음 운영 행동을 넣으면 분석 결과가 의사결정으로 이어진다.

**좋은 인사이트 조건**

| 요소 | 설명 |
|---|---|
| 기준 | 프로그램 유형, 지역, 대상 그룹 등 비교 축이 있음 |
| 숫자 | 1위, 개수, 평균 같은 근거가 있음 |
| 행동 | 확인, 확대, 보완 같은 다음 단계가 있음 |

---

## 문제 15 정답 — 최종 EDA 리포트 결론

In [ ]:
print("프로젝트 대시보드:")
print(project_dashboard)
print()

print("참여자 수 1위 프로그램 유형:", top_program_type)
print("참여자 수 1위 지역:", top_district)
print("효율 점수 1위 프로그램:")
print(top_efficiency_program[["program_id", "district", "program_type", "target_group", "efficiency_score"]])
print("점검 대상 프로그램 수:", len(review_programs))

print("최종 결론 초안:")
print(f"- 전체 {len(df)}개 프로그램에서 총 {df['participants'].sum():,}명이 참여했습니다.")
print(f"- 참여자 수는 {top_program_type} 유형과 {top_district} 지역에 가장 많이 집중되었습니다.")
print(f"- 효율 점수 1위 프로그램은 {top_efficiency_program['program_id']}이며 완료율, 만족도, 비용을 함께 반영했습니다.")
print(f"- 점검 대상은 {len(review_programs)}개로 완료율 또는 만족도 기준 재검토가 필요합니다.")
print("- 다음 운영에서는 규모가 큰 영역과 효율이 높은 영역을 나누어 확대 전략을 세웁니다.")

### 왜 이 코드가 정답인지

최종 결론은 전체 규모, 강점, 효율, 위험, 다음 행동을 한 번에 정리해야 한다. `project_dashboard` 와 앞에서 만든 1위 변수들을 다시 출력하면 결론이 실제 코드 결과와 연결된다. 점검 대상 수까지 포함하면 좋은 사례만 강조하는 보고서가 아니라 개선 과제까지 포함한 프로젝트가 된다.

**모범 결론 예시**

```text
전체 720개 프로그램에서 많은 참여자가 발생했지만, 참여자 수만으로 우수 프로그램을 판단하면 안 된다.
참여자 수가 큰 유형과 지역은 운영 규모 측면에서 중요하다.
효율 점수는 완료율, 만족도, 비용을 함께 반영해 확대 후보를 찾는 보조 기준이다.
점검 대상 프로그램은 완료율 또는 만족도가 낮으므로 세부 원인을 확인해야 한다.
다음 운영에서는 규모 확대 후보와 품질 개선 후보를 분리해 관리한다.
```

---

## 채점 포인트

| 항목 | 확인 기준 |
|---|---|
| 데이터 품질 | 결측, 중복 ID, 값 범위를 확인 |
| KPI | 1인당 비용, 완료자 수, 완료율 구간, 효율 점수 생성 |
| 그룹 분석 | 프로그램 유형, 지역, 대상 그룹 요약 |
| 시각화 | 피벗 히트맵, 산점도, 월별 선 그래프 중 2개 이상 |
| 결론 | 전체 규모, 강점, 점검 대상, 다음 행동 포함 |

## 흔한 오답

- 완료율을 0~100 값으로 착각해 점수를 잘못 만든다.
- 비용 점수에서 높은 비용을 높은 점수로 처리한다.
- 참여자 수 1위만 보고 우수 프로그램이라고 단정한다.
- 피벗 테이블 합계와 원본 합계를 검증하지 않는다.
- 최종 결론에 숫자가 없거나 다음 행동이 없다.

## 교사용 상세 피드백 가이드

학생 답안을 볼 때는 코드가 길어 보이는지보다 프로젝트 흐름이 완성됐는지 본다. 다음 순서로 확인한다.

1. **문제 정의가 보이는가**: "운영 성과 개선"이라는 질문에 답하고 있는지 확인한다. 단순히 표만 많이 만들면 프로젝트 리포트가 아니다.
2. **KPI 방향이 맞는가**: 완료율과 만족도는 높을수록 좋고, 1인당 비용은 낮을수록 좋다. 효율 점수 방향을 틀리면 결론이 반대로 나온다.
3. **그룹 기준이 다양하게 쓰였는가**: 프로그램 유형, 지역, 대상 그룹 중 하나만 보면 운영 판단이 좁아진다.
4. **최종 결론이 실행 가능한가**: "좋다", "나쁘다"에서 끝나지 않고 확대, 점검, 개선 같은 행동으로 이어져야 한다.

## 부분 점수 기준

| 상황 | 처리 |
|---|---|
| 파일 로드와 품질 확인만 성공 | 문제 1~3 일부 통과 |
| KPI 계산은 했지만 방향 오류 | 기준 재설명 후 수정 |
| 그룹 요약은 있으나 결론 없음 | 코드 점수 인정, 리포트 재작성 |
| 시각화만 있고 숫자 검증 없음 | 발표 품질 보완 필요 |
| 최종 결론이 코드와 불일치 | 결론 재제출 |

## 보너스 확장 아이디어

빠른 학생에게는 효율 점수 가중치를 직접 바꿔 보게 한다. 예를 들어 만족도 중심, 비용 중심, 완료율 중심으로 세 가지 점수를 만들고 상위 프로그램이 어떻게 바뀌는지 비교할 수 있다. 또한 분기별 운영 흐름이나 지역별 프로그램 유형 비중을 추가하면 발표용 인사이트가 더 풍부해진다.

## 수업 중 피드백 문장 예시

- “이 기준은 규모를 보는 건가요, 품질을 보는 건가요?”
- “1인당 비용은 낮을수록 좋은데 점수 방향이 맞나요?”
- “이 결론을 보고 운영자가 어떤 행동을 해야 하나요?”
- “참여자 수 1위와 효율 점수 1위가 다르면 어떻게 설명할까요?”

## 모범 답안 사용 주의

모범 답안은 프로젝트 흐름의 예시다. 학생이 다른 가중치로 효율 점수를 만들거나, 다른 기준으로 우수 프로그램을 정의해도 기준을 명확히 설명하고 결론이 코드와 맞으면 인정할 수 있다. 다만 기준이 코드와 결론에서 달라지거나, 숫자 없이 감상문처럼 쓰면 프로젝트 제출물로는 부족하다.

## 재현성 확인

강사는 답안 노트북을 런타임 재시작 후 처음부터 실행해 본다. 통합 프로젝트는 앞 셀에서 만든 파생 열과 요약표가 뒤 셀에 많이 쓰이므로 셀 순서가 중요하다. `df`, `program_summary`, `district_summary`, `target_summary`, `strong_programs`, `review_programs`, `project_dashboard` 가 순서대로 만들어지는지 확인한다.

## 결론 문장 채점 예시

결론이 “프로그램을 분석했다”에서 끝나면 부족하다. “참여자 수는 특정 유형과 지역에 집중되지만, 효율 점수 상위 프로그램은 완료율·만족도·비용을 함께 만족하므로 다음 운영에서는 규모 확대 후보와 품질 개선 후보를 분리해 관리한다”처럼 기준과 행동이 함께 들어가야 한다.